# Poem Sentiment: DSPy Auto Prompt Engineering

**Goal:** Use DSPy to automatically optimize prompts for the local Ministral-3-8B model — no manual prompt crafting.

**Comparison table so far:**
| Model | Accuracy |
|---|---|
| Naive Bayes (TF-IDF) | 66.35% |
| Ministral-3-8B zero-shot (hand-crafted few-shot) | 54.81% |
| Ministral-3-8B QLoRA | 80.77% |
| **DSPy auto-prompt (this notebook)** | **TBD** |

**Dataset:** `google-research-datasets/poem_sentiment`  
**Labels:** `0=negative`, `1=positive`, `2=no_impact`, `3=mixed`

## What DSPy Does
DSPy replaces hand-written prompts with *compiled* programs. The optimizer searches over:
- **Instructions** — the task description / system prompt
- **Few-shot demonstrations** — automatically selected from the training set

The original hand-crafted prompt in `mistral_finetune.ipynb` had 4 fixed examples and a static instruction string. DSPy will find better ones automatically.

## Why a custom LM wrapper?
The local `unsloth/Ministral-3-8B-Base-2512-unsloth-bnb-4bit` is pre-quantized and needs
`Mistral3ForConditionalGeneration` (not a standard pipeline). We wrap it so DSPy can call it
like any other LM.

In [ ]:
# Install dependencies (run once)
# !pip install dspy datasets scikit-learn -q

In [ ]:
import json
import re
from collections import Counter
from typing import Literal

import torch
import dspy
from datasets import load_dataset
from sklearn import metrics
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration

print(f"DSPy version : {dspy.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

## 1. Local LM Wrapper

DSPy expects its LM to be callable: `lm(prompt=str, **kwargs) -> list[str]`.  
We load the local Ministral model exactly as in `mistral_finetune.ipynb` and expose that interface.

In [ ]:
MODEL_ID = "unsloth/Ministral-3-8B-Base-2512-unsloth-bnb-4bit"


class MinistralLocalLM(dspy.LM):
    """
    DSPy LM wrapper for the local pre-quantized Ministral-3-8B model.

    DSPy formats its prompt as a plain text template (compatible with base / completion
    models) and calls __call__(prompt=...) expecting a list[str] of completions back.
    """

    def __init__(self, model_id: str = MODEL_ID, max_new_tokens: int = 64):
        # Skip dspy.LM.__init__ which would call litellm — we handle generation ourselves.
        # We still set the attributes DSPy inspects.
        self.model       = model_id
        self.model_type  = "text"
        self.max_new_tokens = max_new_tokens
        self.history     = []
        self.kwargs      = {"temperature": 0.0, "max_tokens": max_new_tokens, "n": 1}

        print(f"Loading tokenizer from {model_id} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)

        print("Loading model (pre-quantized NF4 weights) ...")
        self.hf_model = Mistral3ForConditionalGeneration.from_pretrained(
            model_id, device_map="auto"
        )
        self.hf_model.config.use_cache = False
        self.hf_model.eval()
        print(f"Ready.  VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    # ------------------------------------------------------------------
    # Core generation
    # ------------------------------------------------------------------

    @torch.inference_mode()
    def _complete(self, text: str, max_new_tokens: int) -> str:
        inputs    = self.tokenizer(text, return_tensors="pt").to(self.hf_model.device)
        prompt_len = inputs["input_ids"].shape[1]
        output_ids = self.hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        return self.tokenizer.decode(
            output_ids[0][prompt_len:], skip_special_tokens=True
        )

    # ------------------------------------------------------------------
    # DSPy LM interface
    # ------------------------------------------------------------------

    def _messages_to_text(self, messages: list[dict]) -> str:
        """Flatten OpenAI-style messages into a plain text string for a base model."""
        return "\n\n".join(m.get("content", "") for m in messages)

    def __call__(self, prompt=None, messages=None, n=1, max_tokens=None, **kwargs):
        text   = self._messages_to_text(messages) if messages else (prompt or "")
        n_toks = max_tokens or self.max_new_tokens

        completions = [self._complete(text, n_toks) for _ in range(n)]

        self.history.append({"prompt": text, "response": completions})
        return completions

    def inspect_history(self, n: int = 1):
        for entry in self.history[-n:]:
            print("=== PROMPT ===")
            print(entry["prompt"])
            print("=== RESPONSE ===")
            for c in entry["response"]:
                print(repr(c))
            print()


lm = MinistralLocalLM(MODEL_ID, max_new_tokens=32)
dspy.configure(lm=lm)

## 2. Load Dataset

In [ ]:
LABEL_MAP = {0: "negative", 1: "positive", 2: "no_impact", 3: "mixed"}

ds = load_dataset("google-research-datasets/poem_sentiment")

for split, subset in ds.items():
    label_counts = Counter(subset['label'])
    readable = {LABEL_MAP[k]: v for k, v in sorted(label_counts.items())}
    print(f"{split:12s} n={len(subset):4d}  {readable}")

In [ ]:
def hf_to_dspy(split) -> list[dspy.Example]:
    return [
        dspy.Example(
            verse=row['verse_text'],
            sentiment=LABEL_MAP[row['label']],
        ).with_inputs('verse')
        for row in split
    ]

train_data = hf_to_dspy(ds['train'])
val_data   = hf_to_dspy(ds['validation'])
test_data  = hf_to_dspy(ds['test'])

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")
print("Sample:", train_data[0])

## 3. DSPy Program

DSPy 3.x defaults to a `JSONAdapter` format (`[[ ## verse ## ]]` markers + "Respond with JSON") which the **base completion model ignores** — it emits EOS immediately because the format is alien to it.

Fix: `SentimentClassifier.forward` builds the prompt manually in the **JSON-completion style** from `mistral_finetune.ipynb` that the model knows:

```
Verse: "..."
Output: {"label": "..."}
...
Verse: "<test>"
Output: {          ← model completes here
```

`self.predict` (a `dspy.Predict`) is kept only so DSPy's optimizers can write their chosen demos to `self.predict.demos`. The `forward` method reads them back and formats them into the prompt.

In [ ]:
FALLBACK_DEMOS = [
    ("and the dead leaves lie huddled and still", "negative"),
    ("what light through yonder window breaks",   "positive"),
    ("the sun sets slowly in the west",           "no_impact"),
    ("a tender joy yet touched with pain",        "mixed"),
]
VALID_LABELS = {"negative", "positive", "no_impact", "mixed"}


def _get(demo, field):
    """Access a field whether demo is a dict (from JSON) or a dspy.Example."""
    return demo[field] if isinstance(demo, dict) else getattr(demo, field)


class PoemSentiment(dspy.Signature):
    """Classify the emotional sentiment of a poem verse."""
    verse: str = dspy.InputField(desc="A line or verse from a poem")
    sentiment: Literal["negative", "positive", "no_impact", "mixed"] = dspy.OutputField(
        desc="One of: negative (grief/sorrow), positive (joy/hope), "
             "no_impact (neutral/descriptive), mixed (ambivalent)"
    )


class SentimentClassifier(dspy.Module):
    def __init__(self):
        self.predict = dspy.Predict(PoemSentiment)

    def _build_prompt(self, verse: str) -> str:
        lines = [
            "Classify the sentiment of each poem verse. "
            "Output only a JSON object with key 'label'. "
            "Valid values: negative, positive, no_impact, mixed.\n"
        ]
        demos = self.predict.demos or []
        examples = [(_get(d, "verse"), _get(d, "sentiment")) for d in demos] \
                   or FALLBACK_DEMOS
        for text, label in examples:
            lines.append(f'Verse: "{text}"')
            lines.append(f'Output: {{"label": "{label}"}}\n')
        lines.append(f'Verse: "{verse}"')
        lines.append('Output: {')
        return "\n".join(lines)

    def _parse(self, completion: str) -> str | None:
        text = ("{" + completion).strip()
        try:
            obj = json.loads(text if text.endswith("}") else text + "}")
            label = obj.get("label", "").lower()
            return label if label in VALID_LABELS else None
        except (json.JSONDecodeError, ValueError):
            pass
        m = re.search(r'"label"\s*:\s*"(\w+)"', text)
        if m:
            label = m.group(1).lower()
            return label if label in VALID_LABELS else None
        for label in VALID_LABELS:
            if label in completion.lower():
                return label
        return None

    def forward(self, verse: str) -> dspy.Prediction:
        prompt      = self._build_prompt(verse)
        completions = lm(prompt=prompt, max_tokens=32)
        raw         = completions[0] if completions else ""
        return dspy.Prediction(sentiment=self._parse(raw))


# Smoke test
classifier = SentimentClassifier()
result = classifier(verse="and the dead leaves lie huddled and still")
print(f"Smoke test — expected=negative  got={result.sentiment!r}")

## 4. Evaluation Metric

In [ ]:
def accuracy_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> bool:
    if not prediction.sentiment:
        return False
    return example.sentiment.lower() == prediction.sentiment.lower()


def evaluate(program, dataset) -> float:
    evaluator = dspy.Evaluate(
        devset=dataset,
        metric=accuracy_metric,
        num_threads=1,
        display_progress=True,
        display_table=False,
        provide_traceback=True,
    )
    score = evaluator(program)
    return score / 100.0


print("Metric defined.")

## 5. Baseline (already measured)

The baseline is the **hand-crafted few-shot prompt** from `mistral_finetune.ipynb`:

```python
# build_prompt() in mistral_finetune.ipynb
# Fixed instruction + 4 hand-picked examples (one per label)
# Output format: JSON completion  →  {"label": "<sentiment>"}
#
# Test-set accuracy: 54.81%
```

DSPy's job is to **beat 54.81%** by automatically selecting better instructions and examples.  
We skip re-running the baseline here and go straight to optimization.

## 6. Auto Prompt Optimization

### Strategy A — BootstrapFewShot
Runs the **teacher model** (our local Ministral) on the training set, collects examples it answered correctly, and injects them as few-shot demonstrations.

- Fast (one pass through train set)
- Directly equivalent to the hand-crafted 4-shot prompt, but chosen automatically

### Strategy B — MIPROv2
Jointly optimizes the **instruction text** AND the **demonstrations** via a Bayesian search.  
More iterations = better prompt, more inference calls.

> **Note:** Both optimizers use `num_threads=1` to avoid CUDA OOM on shared GPU memory.

In [ ]:
# ---- Strategy A: BootstrapFewShot ----
from dspy.teleprompt import BootstrapFewShot

teleprompter_bfs = BootstrapFewShot(
    metric=accuracy_metric,
    max_labeled_demos=8,
    max_bootstrapped_demos=4,
    max_rounds=1,
)

print("Compiling with BootstrapFewShot ...")
optimized_bfs = teleprompter_bfs.compile(
    SentimentClassifier(),
    trainset=train_data,
)
print("Done.")

In [ ]:
bfs_val_acc = evaluate(optimized_bfs, val_data)
print(f"Val accuracy (BootstrapFewShot): {bfs_val_acc:.4f}")

In [ ]:
print("=== Auto-selected demonstrations ===")
for i, demo in enumerate(optimized_bfs.predict.demos):
    print(f"[{i}] {_get(demo, 'verse')!r:55s}  ->  {_get(demo, 'sentiment')}")

In [ ]:
# ---- Strategy B: MIPROv2 ----
from dspy.teleprompt import MIPROv2

teleprompter_mipro = MIPROv2(
    metric=accuracy_metric,
    auto='light',       # ~25 trials; try 'medium' for ~50 if you have time
    num_threads=1,      # single-thread to avoid GPU OOM
    verbose=True,
)

print("Compiling with MIPROv2 (auto='light') ...")
optimized_mipro = teleprompter_mipro.compile(
    SentimentClassifier(),
    trainset=train_data,
    valset=val_data,
    requires_permission_to_run=False,
)
print("Done.")

In [ ]:
mipro_val_acc = evaluate(optimized_mipro, val_data)
print(f"Val accuracy (MIPROv2): {mipro_val_acc:.4f}")

In [ ]:
print("=== MIPROv2 optimized instruction ===")
print(optimized_mipro.predict.signature.instructions)
print()
print("=== Demonstrations ===")
for i, demo in enumerate(optimized_mipro.predict.demos):
    print(f"[{i}] {_get(demo, 'verse')!r:55s}  ->  {_get(demo, 'sentiment')}")

## 7. Final Evaluation on Test Set

In [ ]:
def full_report(program, dataset, name: str) -> float:
    y_true, y_pred = [], []
    for ex in dataset:
        try:
            pred  = program(verse=ex.verse)
            label = pred.sentiment.lower().strip()
            if label not in VALID_LABELS:
                label = "no_impact"      # fallback for malformed output
        except Exception:
            label = "no_impact"
        y_true.append(ex.sentiment)
        y_pred.append(label)

    acc = metrics.accuracy_score(y_true, y_pred)
    print(f"\n{'='*52}")
    print(f"  {name}")
    print(f"  Test accuracy: {acc:.4f}")
    print(f"{'='*52}")
    print(metrics.classification_report(
        y_true, y_pred,
        labels=list(LABEL_MAP.values()),
        zero_division=0,
    ))
    return acc


bfs_test_acc   = full_report(optimized_bfs,   test_data, "BootstrapFewShot")
mipro_test_acc = full_report(optimized_mipro, test_data, "MIPROv2")

## 8. Full Comparison Table

In [ ]:
NAIVE_BAYES_ACC    = 0.6635
MINISTRAL_ZEROSHOT = 0.5481   # hand-crafted 4-shot prompt from mistral_finetune.ipynb
MINISTRAL_QLORA    = 0.8077

rows = [
    ("Naive Bayes (TF-IDF)",                    NAIVE_BAYES_ACC),
    ("Ministral-3-8B zero-shot (hand-crafted)",  MINISTRAL_ZEROSHOT),
    ("Ministral-3-8B QLoRA",                     MINISTRAL_QLORA),
    ("DSPy BootstrapFewShot",                     bfs_test_acc),
    ("DSPy MIPROv2",                              mipro_test_acc),
]

rows_sorted = sorted(rows, key=lambda x: x[1], reverse=True)

print(f"\n{'Model':<48} {'Accuracy':>10}")
print("-" * 60)
for name, acc in rows_sorted:
    print(f"{name:<48} {acc:>10.4f}")
print("-" * 60)
print()
print("DSPy gain over hand-crafted zero-shot (54.81%):")
print(f"  BootstrapFewShot : {bfs_test_acc  - MINISTRAL_ZEROSHOT:+.4f}")
print(f"  MIPROv2          : {mipro_test_acc - MINISTRAL_ZEROSHOT:+.4f}")

## 9. Save Optimized Programs

In [ ]:
optimized_bfs.save("dspy_optimized/dspy_bfs_optimized.json")
optimized_mipro.save("dspy_optimized/dspy_mipro_optimized.json")
print("Saved: dspy_optimized/dspy_bfs_optimized.json, dspy_optimized/dspy_mipro_optimized.json")

# Reload example:
# loaded = SentimentClassifier()
# loaded.load('dspy_optimized/dspy_mipro_optimized.json')